#### Bounding boxes

### What is Object recognition?
Object recognition idetifies objects in images:
- Location of each object in an image (Bounding box)
- Class label of each object

Applications: Surveillance, medical diagnosis, traffic management, sprts analytics


#### Bounding box representation
- A rectangular box describing the object's spatial location
- Training data annotation & model outputs
- Ground truth bounding box: precise object location
- Bounding box coordinates:
    - Top left and bottom right
    - Bounding box = (x1, y1, x2, y2)
    - x1 = x_min, x2 = x_max, ...

#### Converting Pixels to tensors
Transforming with `ToTensor()`

- Tensor type:
    - torch.float
- Scaled tensor range:
    - [0.0, 1.0]

In [3]:

from PIL import Image

img = Image.open('expresso.png')


In [4]:
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()
])

image_tensor = transform(img)


Transformingb with `PILToTensor()`
- Tensor type:
    - torch.uint8(8-bit integer)
- Unscaled tensor range:
    - [0, 255]

In [5]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.PILToTensor()
])
image_tensor = transform(img)

### Drawing the bounding box
- import `draw_bounding_boxes`
- Collect coordinates into a tensor
- Unsqueeze to two dimensions
- Transform to image and plot

In [6]:
# Define sample bounding box coordinates (replace with your actual object's coordinates)
x_min, y_min, x_max, y_max = 50, 50, 250, 250 

In [15]:
import torch
from PIL.ImageEnhance import Color
from torchvision.utils import draw_bounding_boxes

bbox = torch.tensor([x_min, y_min, x_max, y_max])
bbox = bbox.unsqueeze(0)
bbox_image = draw_bounding_boxes(
    image_tensor, bbox, width=3, colors='red'
)

transform = transforms.Compose([
    transforms.ToPILImage()
])
pil_image = transform(bbox_image)

import matplotlib.pyplot as plt 
plt.imshow(pil_image)

: 

### Image Tensor

In [8]:
import torch
# Convert bbox into tensors
bbox_tensor = torch.tensor(bbox)

# Add a new batch dimension
bbox_tensor = bbox_tensor.unsqueeze(0)

# Resize image and transform tensor
transform = transforms.Compose([
  transforms.Resize(224),
  transforms.PILToTensor()
])

# Apply transform to image
image_tensor = transform(img)
print(image_tensor)

NameError: name 'bbox' is not defined

In [10]:
# Import draw_bounding_boxes
import matplotlib
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

# Define the bounding box coordinates
bbox = [ x_min, y_min, x_max, y_max]
bbox_tensor = torch.tensor(bbox).unsqueeze(0)

# Implement draw_bounding_boxes
img_bbox = draw_bounding_boxes(image_tensor, bbox_tensor, width=3, colors="red")

# Tranform tensors to image
transform = transforms.Compose([
    transforms.ToPILImage()
])
plt.imshow(transform(img_bbox))
plt.show()

: 

#### Evaluating object recognition models

##### Intersection over union (IoU)
- Object of interest: object in image we want to detect (e.g., dog)
- Ground truth box: the accurate bounding box around the object of interest
- Intersection over Union: a metric to measure the overlab between two boxes
- IoU = Area of intersection / Area of Union
    - IoU = 0 no overlap, IoU = 1 perfect overlap
    - ``IoU > 0.5 is a good prediction``

#### IoU in PyTorch


In [2]:
import torch
bbox1= [50, 50, 150, 150]
bbox2 = [100, 100, 200, 200]


bbox1 = torch.tensor(bbox1).unsqueeze(0)
bbox2 = torch.tensor(bbox2).unsqueeze(0)


In [3]:
from torchvision.ops import box_iou

iou = box_iou(bbox1, bbox2)
print(iou)

tensor([[0.1429]])


#### Predicting bounding boxes

In [ ]:
model.eval()
with torch.no_grad():
    output = model(input_image)
print(output)

In [ ]:
boxes = output[0]['boxes']
# Confidence score
score = output[0]['scores']

#### Non-max suppression (NMS)
Non-max suppression: a common technique to select the most relevant bounding boxes
- Non-max: discarding boxes with low confidence scoe to xcontain an object
- Suppression: discarding boxes with low IoU

#### Non-max suppression in PyTroch
- Boxes: tensors with the bounding box coordinates of the shape [N, 4]
- Scores: tensor with the confidence score for each box of the shape [N]
- iou_threshold: the threshold  between 0.0 and 1.0

In [ ]:
from torchvision.ops import nms

box_indices = nms(
    boxes=boxes,
    scores=scores,
    iou_threshold=0.5
) 

print(box_indices)

### Bounding boxes prediction

In [ ]:
# Get model's prediction
with torch.no_grad():
    output = model(test_image)

# Extract boxes from the output
boxes = output[0]['boxes']

# Extract scores from the output
scores = output[0]['scores']

print(boxes, scores)

#### Calculate NMS

In [ ]:
# Import nms
from torchvision.ops import nms

# Set the IoU threshold
iou_threshold = 0.5
# Apply non-max suppression
box_indices = nms(
    boxes=boxes,
    scores=scores,
    iou_threshold=iou_threshold
)


# Filter boxes
filtered_boxes = boxes[box_indices]

print("Filtered Boxes:", filtered_boxes)

### Object detection using R-CNN

#### Region-based CNN family: R-CNN
R-CNN family: R-CNN, Fast-CNN, Fast CNN
- Module 1: generation of region proposals
- Module 2: feature extraction (convolutional layers)
- Module 3: class and bounding box prediction

R-CNN: backbone
- Convolutional laayers: pre-trained models
    - Backbone: the core CNN architecture responsible for feature extraction
- Convolutional & Pooling layers
- Extract features for region proposals and object detection

#### R-CNN: backbone with PyTorch
- `.features`: only convolutional layers
- `.children()`: all layers from block
- `nn.Sequential(*list())`: all sub-layers are placed into a sequential block as a list 
    - `*`: unpacks the elements from list

In [6]:
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

vgg = vgg16(weights=VGG16_Weights.DEFAULT)

backbone = nn.Sequential(
    *list(vgg.features.children())
)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [21:47<00:00, 423kB/s]    


### R-CNN: classifier layer
- Extract backbone's output size


In [ ]:
input_dimension = nn.Sequential(*list(
    vgg_backbone.classifier.children())
    )[0].in_features

- Create a new classifier

In [ ]:
classifier = nn.Sequential(
    nn.Linear(input_dimension, 512),
    nn.ReLU(),
    nn.Linear(512, num_classes),
)

#### R-CNN: box regressor layer
- Sits on top of the backbone
- 4 outputs for 4 box coordinates

In [ ]:
box_regressor = nn.Sequential(
    nn.Linear(input_dimension, 32),
    nn.ReLU(),
    nn.Linear(32, 4),
)

#### Putting it all together: object detection model

In [8]:
class ObjectDetectorCNN(nn.Module):
    def __init__(self):
        super(ObjectDetectorCNN, self).__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(vgg.features.children()))
        input_features = nn.Sequential(*list(vgg.classifier.children()))[0].in_features
        self.classifier = nn.Sequential(
            nn.Linear(input_features, 512),
            nn.ReLU(),
            nn.Linear(512, 2),
        )
        self.box_regressor = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU(),
            nn.Linear(32, 4),
        )

        def forward(self, x):
            features = self.backbone(x)
            bboxes = self.regressor(features)
            classes = self.classifier(features)
            return bboxes, classes


### Running object recognition
1. Load and transform the image
2. `unsqueeze()` the image to add the batch dimension
3. Pass the image tensor to the model
4. Run Non-Max Suppression (`nms()`) over model
5. `draw_bounding_boxes()` on top of the image

### Exercise:
- Pre-trained model backbone

In [ ]:
# Load pretrained weights
vgg_model = vgg16(weights=VGG16_Weights.DEFAULT)

# Extract the input dimension
input_dim = nn.Sequential(*list(vgg_model.classifier.children()))[0].in_features

# Create a backbone with convolutional layers
backbone = nn.Sequential(*list(vgg_model.features.children()))

# Print the backbone model
print(backbone)